## constructing galois field of characteristic '2'
Scinse most of our src information is in binaries we need to construct extended field of characteristic 2 which is: $GF(2^m)$ and is constructed by a polynomial. here i made a function called `exfield_gen` that does that, also for operation powers of elements also needed scince `index()` method can be slow i saved powers of primitive elements too in list `loga` this makes *multiplication*, *division* and *power* operators in $GF(2^m)$ easy. I ordered bits **LSB** to **MSB** to match the textbook notation. 

```python
def exfield_gen(m, g):
	n =(1 << m)
	x =(1 << m)
	alph =[0] * (n-1)
	loga =[0] * (n)

	for i in range(n -1):
		x = (x >> 1)
		alph[i] = x
		loga[x] = i

		if(x & 1):
			x = (x ^ g)

    return alph, loga
```
As you can see here i just multiply it by $\alpha$ and subtracting by $g(x)$ which is a minimal polynomial.

Above function gives the field elements and thier corresponding powers, which can be used to calculate *multiplication* or *division*. (addition is just normal *xor* scince field is a characteristic 2).

In [5]:
import gf

gf16 = gf.exfield_gen(4, 0b11001)
alph = gf16[0]
loga = gf16[1]

print(f"primitive elements (indexs are powers): {tuple(bin(i) for i in alph)} \n primitive elements powers: {tuple(loga)}")

primitive elements (indexs are powers): ('0b1000', '0b100', '0b10', '0b1', '0b1100', '0b110', '0b11', '0b1101', '0b1010', '0b101', '0b1110', '0b111', '0b1111', '0b1011', '0b1001') 
 primitive elements powers: (0, 3, 2, 6, 1, 9, 5, 11, 0, 14, 8, 13, 4, 7, 10, 12)


## Cyclic code encoder:
I wrote a **LFSR** hardware model function (per cycle) to simulate encoder used in Cyclic codes and binary **BCH** codes.

```python
def lfsr(taps :int, mem :int =0, /, state :int =0, sin :bool =0):
	gate1 = (state & 1)
	state |= (sin << mem)
	if(gate1):
		state ^= taps
	else:
		state  = 0
		
	state >>= 1

	return state
```
Above code is a model based on the example(4.2) in shun li textbooks, where `gate1` indicates feedback input.

For encoder simply run `massage`, $u(x)$, $k$ times then output the state of `lfsr`:

```python
def encode(n, k, /, u, g, parity =0):
    
	codeword =0b0
	taps = g ^ (g & 1)
	lfsr_par = (taps, n-k)
	# clock through n _codeword length_
	for clk in range(n):
		u_clk, u =cir.bit_pop(u)

		parity = cir.lfsr(*lfsr_par, state =parity, sin =u_clk)
		if(clk < k):
			codeword ^= (u_clk << clk)
	codeword ^= (parity <<k)

	return codeword
```


## Berlekamp Massey
This algorithm used to solve the linear recursion, its like power estimation:

![bm flowchart](./img/bmflchart.jpg)

Consider linear recursion as following:
$$
S_k =\sum_{i}^{n-1} \sigma_i.S_{k-i}
$$
Berlekamp massey says to find values of $\Lambda_i$, if there is discrepancy, you need to take off some multiple of last step connection polynomial $\sigma^{(\rho)}(X)$ from $\sigma^{(\mu)}(X)$, because of linearaty we can say that multiple $A$ is calculated as follows: $\Delta_\mu +A.\Delta_\rho =0 \to A = -\frac{\Delta_\mu}{\Delta_\rho}$.
here I took $\sigma^{\rho}(X) =B(X)$ and $\sigma^{\mu}(X) =C(X)$ in each iteration. ...

## Code:
First define a field:

In [26]:
import gf
field =gf.exfield_gen(4, 0b11001) # extened field
expo  =field[1]
alph  =field[0]

# form textbook's example 5.3:
S = [alph[0], alph[0], alph[10], alph[0], alph[10], alph[5]]

# pre-loop var initializaton:
#current conncetion,  last connection,  last discrepancy,  length of connection,   prev step
C =[alph[0]];       B =[alph[0]];    dr =1;            l =0;                  rho =-1

## Euclidean algorithm:

In [ ]:
def Euc(a, q, x =0):

    x -= 1
    return Euc(a, q-a, x)
    print(a, q, q-a)
    if (q-a):
        return x

print(Euc(3, 7))

RecursionError: maximum recursion depth exceeded